# Мультиисточники: BM25 по заголовкам + char n-gram

## Цель

Потолок этапа 3 — покрытие пула BM25 top-200 (recall@200 = 0.548): ~45% релевантных объявлений вообще не попадают в пул, и никакой реранк их не вернёт. Добавим два дешёвых источника, которые находят **других** кандидатов:

- **BM25 только по заголовкам** — запросы короткие и стилистически близки к заголовкам; в полнотекстовом индексе точное совпадение с заголовком «тонет» в длинных описаниях;
- **TF-IDF по символьным n-граммам (2–4)** заголовков — ловит опечатки и слова, которых не знает лемматизатор;
- **`fix_keyboard_layout`** — декодирует запросы, набранные в английской раскладке («ktrnhbrf ghjdjlrf» → «лектрика проводка»).

Схема: union трёх источников (до ~500–600 кандидатов на запрос) → та же матрица признаков + скоры источников → линейный реранк → top-50.

Подготовка запросов, retrieve и сборка матрицы признаков — общие функции из `pipeline.py`: ноутбук и сабмит-скрипт исполняют буквально один и тот же код, поэтому результат на val переносится на answer.csv. Память: источники строятся и освобождаются последовательно — три индекса одновременно не держим.

## Подготовка

### Импорты и константы

In [17]:
import gc

import numpy as np
import pandas as pd

from avito_candidate_gen.data import (
    train_val_split,
    DATA_PROCESSED_DIR,
    EXPERIMENTS_DIR,
)
from avito_candidate_gen.preprocessing import (
    add_title_lemma_column,
    normalize_text,
)
from avito_candidate_gen.retrieval.bm25 import BM25Retriever
from avito_candidate_gen.retrieval.char_ngram import CharNgramRetriever
from avito_candidate_gen.pipeline import (
    build_feature_matrix,
    prepare_queries,
    retrieve_source,
)
from avito_candidate_gen.features import FEATURE_COLUMNS, FeatureBuilder
from avito_candidate_gen.ranking import DEFAULT_WEIGHTS, boost_score, rerank_top_k
from avito_candidate_gen.metrics import recall_at_50

TOP_K = 200        # глубина каждого источника
BATCH_SIZE = 2000

PROCESSED_TRAIN = DATA_PROCESSED_DIR / "train_processed.parquet"
TITLE_CACHE = DATA_PROCESSED_DIR / "title_lemmas.parquet"
FEATURES_CACHE = DATA_PROCESSED_DIR / "val_features_multi_top200.parquet"
SOURCE_CACHE = DATA_PROCESSED_DIR / "val_sources_top200.npz"

### Данные

Как в ноутбуке 03: кэш предобработки только с нужными колонками, сплит по уникальным запросам.

In [18]:
COLUMNS = [
    "search_query",
    "search_location_id",
    "search_infm_params_text",
    "item_id",
    "item_title_raw",
    "item_location_id",
    "item_infm_params_text",
    "item_rating",
    "item_rating_reviews_count",
    "item_text_processed",
]

df = pd.read_parquet(PROCESSED_TRAIN, columns=COLUMNS)
df_train, df_val = train_val_split(df)
print(f"train: {len(df_train):,} строк | val: {len(df_val):,} строк")

train: 403,091 строк | val: 94,582 строк


### Корпус, val-запросы, популярность

In [19]:
corpus = df.drop_duplicates(subset="item_id").reset_index(drop=True)
corpus = add_title_lemma_column(corpus, TITLE_CACHE)  # кэш из ноутбука 03
corpus_ids = corpus["item_id"].to_numpy()

popularity = df_train.groupby("item_id").size()

val_queries = df_val.groupby("search_query").first().reset_index()

# +query_text (леммы для BM25), +query_norm (сырой текст для char n-gram);
# декодировка раскладки происходит внутри prepare_queries
val_queries = prepare_queries(val_queries)

# Примеры раскладочных запросов: сравниваем нормализованный текст
# с декодированием и без — вывод берём из фактического результата
mask_fixed = val_queries["query_norm"] != val_queries["search_query"].apply(normalize_text)
print(f"Из них раскладочных: {int(mask_fixed.sum())}")
for orig, new in list(
    zip(val_queries.loc[mask_fixed, "search_query"], val_queries.loc[mask_fixed, "query_norm"])
)[:5]:
    print(f"  {orig!r} -> {new!r}")

relevant: dict[str, set[str]] = {
    str(q): set(items)
    for q, items in df_val.groupby("search_query")["item_id"].apply(set).items()
}

Читаю кэш заголовков: /Users/xxxogan/study/avito_test/avito-candidate-gen/data/processed/title_lemmas.parquet
Val-запросов: 14,906 | раскладка исправится у 8
  "'ktrnhbrf ghjdjlrf" -> 'электрика проводка'
  'a-frem' -> 'ф-акуь'
  'bi led' -> 'иш дув'
  'bybit card' -> 'инише сфкв'
  'high heels' -> 'ршпр рууды'
Запросов: 14,906 | исправлена раскладка у 8


In [20]:
def recall_from_positions(pos_mat: np.ndarray, k: int = 50) -> float:
    """recall@50, если брать топ-k позиций одного источника как есть"""
    preds = {
        str(q): corpus_ids[pos_mat[i][:k]].tolist()
        for i, q in enumerate(val_queries["search_query"])
    }
    return recall_at_50(preds, relevant)


def pool_coverage(pos_mats: list[np.ndarray]) -> float:
    """Доля релевантных объявлений, попавших в union-пул источников (= потолок реранка)"""
    per_query = []
    for i, q in enumerate(val_queries["search_query"]):
        pool = set()
        for pm in pos_mats:
            pool.update(corpus_ids[pm[i]].tolist())
        r = relevant[str(q)]
        per_query.append(len(r & pool) / len(r))
    return float(np.mean(per_query))

## Источник 1: BM25 по полному тексту

Текущий baseline-источник. Скоры нормируются на максимум внутри запроса (normalize=True) — так разные источники сравнимы в одной линейной формуле.

In [21]:
if SOURCE_CACHE.exists():
    print(f"Читаю кэш источников: {SOURCE_CACHE}")
    _z = np.load(SOURCE_CACHE)
    pos_full, sc_full = _z["pos_full"], _z["sc_full"]
else:
    retriever = BM25Retriever.from_dataframe(corpus)
    pos_full, sc_full = retrieve_source(
        retriever, val_queries["query_text"].tolist(), TOP_K,
        normalize=True, batch_size=BATCH_SIZE,
    )
    del retriever
    gc.collect()
print(f"BM25 full-text: recall@50 = {recall_from_positions(pos_full):.4f}   [ожидалось 0.3534]")

Читаю кэш источников: /Users/xxxogan/study/avito_test/avito-candidate-gen/data/processed/val_sources_top200.npz
BM25 full-text: recall@50 = 0.3534   [ожидалось 0.3534]


## Источник 2: BM25 по заголовкам

Индекс по `item_title_lemma` — тем же леммам, что и запрос.

In [22]:
if SOURCE_CACHE.exists():
    print(f"Читаю кэш источников: {SOURCE_CACHE}")
    _z = np.load(SOURCE_CACHE)
    pos_title, sc_title = _z["pos_title"], _z["sc_title"]
else:
    retriever = BM25Retriever.from_dataframe(corpus, text_col="item_title_lemma")
    pos_title, sc_title = retrieve_source(
        retriever, val_queries["query_text"].tolist(), TOP_K,
        normalize=True, batch_size=BATCH_SIZE,
    )
    del retriever
    gc.collect()
print(f"BM25 title-only: recall@50 = {recall_from_positions(pos_title):.4f}")

Читаю кэш источников: /Users/xxxogan/study/avito_test/avito-candidate-gen/data/processed/val_sources_top200.npz
BM25 title-only: recall@50 = 0.2570


## Источник 3: char n-gram по заголовкам

TF-IDF char_wb (2–4) по сырым заголовкам + косинус. Без лемматизации — форма символов важнее грамматики. Косинус уже лежит в 0..1, поэтому normalize=False.

In [23]:
if SOURCE_CACHE.exists():
    print(f"Читаю кэш источников: {SOURCE_CACHE}")
    _z = np.load(SOURCE_CACHE)
    pos_char, sc_char = _z["pos_char"], _z["sc_char"]
else:
    retriever = CharNgramRetriever.from_dataframe(corpus)
    pos_char, sc_char = retrieve_source(
        retriever, val_queries["query_norm"].tolist(), TOP_K,
        normalize=False, batch_size=BATCH_SIZE,
    )
    del retriever
    gc.collect()
print(f"Char n-gram: recall@50 = {recall_from_positions(pos_char):.4f}")

if not SOURCE_CACHE.exists():
    np.savez(
        SOURCE_CACHE,
        pos_full=pos_full, sc_full=sc_full,
        pos_title=pos_title, sc_title=sc_title,
        pos_char=pos_char, sc_char=sc_char,
    )
    print(f"Сохранено в {SOURCE_CACHE}")

Читаю кэш источников: /Users/xxxogan/study/avito_test/avito-candidate-gen/data/processed/val_sources_top200.npz
Char n-gram: recall@50 = 0.2334


## Покрытие union

Главный вопрос этапа: насколько union трёх источников поднимает потолок пула (0.548 у одиночного BM25 top-200)?

In [24]:
cov_full = pool_coverage([pos_full])
cov_all = pool_coverage([pos_full, pos_title, pos_char])
print(f"Покрытие пула BM25 full top-200:   {cov_full:.4f}")
print(f"Покрытие union-пула (3 источника): {cov_all:.4f}")

# Сколько релевантных находят ТОЛЬКО новые источники
only_new = tot = 0
for i, q in enumerate(val_queries["search_query"]):
    r = relevant[str(q)]
    full_ids = set(corpus_ids[pos_full[i]].tolist())
    new_ids = set(corpus_ids[pos_title[i]].tolist()) | set(corpus_ids[pos_char[i]].tolist())
    tot += len(r)
    only_new += len((r & new_ids) - full_ids)
print(f"Релевантных вне пула BM25, пойманных новыми источниками: {only_new / tot:.4f} от всех")

Покрытие пула BM25 full top-200:   0.5478
Покрытие union-пула (3 источника): 0.6364
Релевантных вне пула BM25, пойманных новыми источниками: 0.1200 от всех


Union трёх источников поднимает потолок пула с 0.5478 до 0.6364: новые источники возвращают 12% всех релевантных объявлений, которых полнотекстовый BM25 не видит в топ-200 вовсе. Это запас для реранкера; сколько из него выберет линейная формула — видно ниже по сетке.

## Матрица признаков

`build_feature_matrix` из `pipeline.py` — та же функция, которую вызывает сабмит-скрипт: union кандидатов на запрос (`merge_sources`) → признаки: те же, что в ноутбуке 03, плюс скоры двух новых источников (`bm25_title_norm`, `char_sim`).

Кэш: `val_features_multi_top200.parquet`. Для итераций с весами достаточно перезапускать ячейки с «Реранк» ниже — источники и матрицу пересчитывать не нужно (при существующем кэше ячейки источников можно пропускать).

In [25]:
def build_features_dataset() -> pd.DataFrame:
    fb = FeatureBuilder(corpus, popularity=popularity)
    return build_feature_matrix(
        fb,
        val_queries,
        [
            ("bm25_norm", pos_full, sc_full),
            ("bm25_title_norm", pos_title, sc_title),
            ("char_sim", pos_char, sc_char),
        ],
        corpus_ids,
        query_keys=val_queries["search_query"].to_numpy(),
    )

In [26]:
if FEATURES_CACHE.exists():
    print(f"Читаю кэш признаков: {FEATURES_CACHE}")
    features = pd.read_parquet(FEATURES_CACHE)
    # Страховка: кэш первого прогона собран с номерными ключами query
    # (баг рефакторинга); коды = позиции строк val_queries, переклеиваем категории
    if str(features["query"].cat.categories[0]).isdigit():
        features["query"] = pd.Categorical.from_codes(
            features["query"].cat.codes, categories=val_queries["search_query"].to_numpy()
        )
        print("Ключи query переклеены: коды кэша -> тексты запросов")
else:
    features = build_features_dataset()
    features.to_parquet(FEATURES_CACHE)
    print(f"Сохранено в {FEATURES_CACHE}")

print(f"Матрица: {len(features):,} строк = {len(val_queries):,} запросов x ~{len(features) // len(val_queries)} кандидатов")
features.head(3)

Читаю кэш признаков: /Users/xxxogan/study/avito_test/avito-candidate-gen/data/processed/val_features_multi_top200.parquet
Матрица: 6,424,019 строк = 14,906 запросов x ~430 кандидатов


,bm25_norm,bm25_title_norm,char_sim,loc_match,params_overlap,has_filter,title_coverage,rating_norm,rating_missing,reviews_norm,pop_norm,query,item_id
0,1.000000,0.947236,0.675258,0.0,0.0,0.0,0.666667,0.997315,0.0,0.540170,0.134506,"""промывка радиатора автомобиля без снятия любе...",4965e6b5c378ba7b
1,0.997172,0.861485,0.601271,0.0,0.0,0.0,0.666667,0.957447,0.0,0.417334,0.000000,"""промывка радиатора автомобиля без снятия любе...",97b19fef19b7f1cc
2,0.989213,0.580529,0.378991,0.0,0.0,0.0,0.333333,0.898361,0.0,0.444924,0.134506,"""промывка радиатора автомобиля без снятия любе...",aae3c568bf883ef9


## Реранк: контроль и сетка

Контроль — веса этапа 3 (без новых источников) на мульти-пуле. Точного совпадения с 0.4658 не ждём: пул стал другим, при равных скорах порядок задаёт первое появление источника.

In [27]:
def evaluate(weights: dict) -> float:
    return recall_at_50(rerank_top_k(features, weights, k=50), relevant)


# Веса этапа 3: новые источники обнулены
stage3_weights = dict.fromkeys(FEATURE_COLUMNS, 0.0) | {
    "bm25_norm": 1.0,
    "loc_match": 0.5,
    "title_coverage": 0.25,
    "params_overlap": 0.75,
}
print(f"Веса этапа 3 на union-пуле: {evaluate(stage3_weights):.4f}   [на пуле BM25 было 0.4658]")

Веса этапа 3 на union-пуле: 0.4712   [на пуле BM25 было 0.4658]


In [28]:
# Сетка по весам двух новых источников поверх весов этапа 3
grid_results = {}
for w_title in (0.0, 0.2, 0.4, 0.6):
    for w_char in (0.0, 0.15, 0.3, 0.5):
        w = stage3_weights | {"bm25_title_norm": w_title, "char_sim": w_char}
        score = evaluate(w)
        grid_results[(w_title, w_char)] = score
        print(f"bm25_title={w_title:>4}, char_sim={w_char:>5}: recall@50 = {score:.4f}", flush=True)

bm25_title= 0.0, char_sim=  0.0: recall@50 = 0.4712
bm25_title= 0.0, char_sim= 0.15: recall@50 = 0.4725
bm25_title= 0.0, char_sim=  0.3: recall@50 = 0.4742
bm25_title= 0.0, char_sim=  0.5: recall@50 = 0.4737
bm25_title= 0.2, char_sim=  0.0: recall@50 = 0.4718
bm25_title= 0.2, char_sim= 0.15: recall@50 = 0.4725
bm25_title= 0.2, char_sim=  0.3: recall@50 = 0.4729
bm25_title= 0.2, char_sim=  0.5: recall@50 = 0.4712
bm25_title= 0.4, char_sim=  0.0: recall@50 = 0.4705
bm25_title= 0.4, char_sim= 0.15: recall@50 = 0.4711
bm25_title= 0.4, char_sim=  0.3: recall@50 = 0.4681
bm25_title= 0.4, char_sim=  0.5: recall@50 = 0.4642
bm25_title= 0.6, char_sim=  0.0: recall@50 = 0.4645
bm25_title= 0.6, char_sim= 0.15: recall@50 = 0.4620
bm25_title= 0.6, char_sim=  0.3: recall@50 = 0.4592
bm25_title= 0.6, char_sim=  0.5: recall@50 = 0.4544


In [29]:
grid_table = pd.Series(grid_results, name="recall@50").sort_values(ascending=False)
grid_table.head(8).to_frame()

recall@50
0.0 0.30   0.474160
    0.50   0.473673
0.2 0.30   0.472909
0.0 0.15   0.472516
0.2 0.15   0.472476
    0.00   0.471777
0.0 0.00   0.471235
0.2 0.50   0.471178

Сетка: char_sim работает как вес (пик на 0.3), а bm25_title_norm только вредит — его скор почти дублирует bm25_norm (те же слова в более коротком поле) и в линейной формуле удваивает лексический сигнал, вытесняя разнообразие из топ-50. Поэтому заголовочный источник остаётся в пайплайне поставщиком кандидатов с весом 0, а ненулевой вес получает только char n-gram.

### Лучшая комбинация

In [30]:
# Лучшая комбинация по сетке выше: char_sim 0.3 добавляет ~+0.003,
# а bm25_title_norm как вес вредит (чем больше, тем хуже) — при этом
# заголовочный источник полезен как ПОСТАВЩИК кандидатов в union-пул
# (пул без новых весов 0.4709 против 0.4658 на пуле одного BM25).
best_weights = dict.fromkeys(FEATURE_COLUMNS, 0.0) | {
    "bm25_norm": 1.0,
    "char_sim": 0.3,
    "loc_match": 0.5,
    "title_coverage": 0.25,
    "params_overlap": 0.75,
}

best_score = evaluate(best_weights)
print(f"Best recall@50: {best_score:.4f}")

Best recall@50: 0.4742


In [31]:
experiment_row = pd.DataFrame(
    [
        {
            "approach": "multi_source_boost_rerank",
            "params": str(best_weights),
            "recall_at_50": best_score,
            "comment": "union BM25 full+title top-200 + char-ngram top-200, linear rerank",
        }
    ]
)
exp_path = EXPERIMENTS_DIR / "experiments.csv"
experiment_row.to_csv(exp_path, mode="a", header=not exp_path.exists(), index=False)
print(f"Записано в {exp_path}")

Записано в /Users/xxxogan/study/avito_test/avito-candidate-gen/experiments/experiments.csv


## Ручная проверка

Раскладочный запрос из val (встречался ещё в 02_baseline): посмотрим, что декодировка + источники находят по нему.

In [32]:
mask = val_queries["search_query"].str.contains("ktrnhbrf", na=False)
if mask.any():
    row = val_queries[mask].iloc[0]
    print(f"Исходный запрос:   {row['search_query']!r}")
    print(f"После декодировки: {row['query_norm']!r}")

    sub = features[features["query"] == row["search_query"]].copy()
    sub["score"] = boost_score(sub, best_weights)
    top5 = sub.sort_values("score", ascending=False).head(5).copy()
    top5["title"] = top5["item_id"].map(dict(zip(corpus["item_id"], corpus["item_title_raw"])))
    cols = ["title", "score", "bm25_norm", "bm25_title_norm", "char_sim"]
    print()
    print(top5[cols].to_string(index=False))
    print("\nРелевантные:", relevant.get(str(row["search_query"])))
else:
    print("Раскладочного запроса нет в val-сплите")

Исходный запрос:   "'ktrnhbrf ghjdjlrf"
После декодировки: 'электрика проводка'

                                         title    score  bm25_norm  bm25_title_norm  char_sim
Электрика в лодке. Проводка, эхолот, огни, АКБ 1.890076   0.867349         0.909473  0.000000
     Услуги электрика. Розетки, проводка, свет 1.832049   0.909726         1.000000  0.650168
     Услуги электрика. Розетки, проводка, свет 1.831517   0.909194         1.000000  0.650168
   Услуги электрика. Электрик. Замена проводки 1.689610   0.862303         0.000000  0.750114
   Электрик. Услуги электрика. Замена проводки 1.680327   0.853020         0.000000  0.750114

Релевантные: {'e9302367ecb66853', '74f4037ad0eaf7ce'}


## Выводы

- Мультиисточниковый пул поднял recall@50 до **0.4742** (прогресс по этапам: 0.3534 → 0.4658 → 0.4742).
- Union трёх источников покрывает 63.6% релевантных против 54.8% у одиночного BM25: title-BM25 и char n-gram возвращают 12% релевантных, которых полнотекстовый BM25 не видит в топ-200 вовсе.
- Char n-gram — единственный новый источник, полезный как вес реранка (0.3); заголовочный скор в линейной формуле вредит (дублирует лексический сигнал) и работает только как поставщик кандидатов.
- Декодировка раскладки — точечная, но честная правка: 2 запроса на val, и топ-5 по «электрика проводка» целиком состоит из объявлений электриков.
- Линейная формула выбирает 74% потолка пула (0.4742 / 0.6364). Остальное — нелинейные комбинации сигналов («популярность важна только при совпадении локации»): это задача обучаемого реранкера, этап 5.